# Steup

# Batch benchmark setup

This section discovers dataset files under `datasets/`, loads one file for a quick timed run (default 10 examples), measures per-example latency calling the LangPro API, and estimates how long the entire file (and all dataset files) would take.

In [2]:
# List dataset files and pick one for a test run
import os
DATASET_DIR = "datasets"
dataset_files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith("_dataset.json")])
print(f"Found {len(dataset_files)} dataset files")
for i,fn in enumerate(dataset_files[:10]):
    print(i+1, fn)

# Choose index 0 by default (adjustable)
TEST_FILE_INDEX = 0
test_file = dataset_files[TEST_FILE_INDEX]
test_path = os.path.join(DATASET_DIR, test_file)
print('\nSelected for benchmark:', test_file)

Found 151 dataset files
1 disjunctive_syllogism-2-0-base-real_dataset.json
2 disjunctive_syllogism-2-0-complex_predicates-real_dataset.json
3 disjunctive_syllogism-2-0-de_morgan-real_dataset.json
4 disjunctive_syllogism-2-0-negation-real_dataset.json
5 gen_contraposition-2-0-base-real_dataset.json
6 gen_contraposition-2-0-complex_predicates-real_dataset.json
7 gen_contraposition-2-0-de_morgan-real_dataset.json
8 gen_contraposition-2-0-negation-real_dataset.json
9 gen_contraposition-2-1-base-real_dataset.json
10 gen_contraposition-2-1-complex_predicates-real_dataset.json

Selected for benchmark: disjunctive_syllogism-2-0-base-real_dataset.json


In [3]:
# Load the test dataset and preview
import json
with open(test_path, 'r', encoding='utf-8') as f:
    items = json.load(f)

print('Total examples in file:', len(items))
print('Sample item keys:', list(items[0].keys()))
print('\nFirst example:')
from pprint import pprint
pprint(items[0])

Total examples in file: 200
Sample item keys: ['P1', 'P2', 'C']

First example:
{'C': 'It is true that Every member of IKBKG deficiency causes anhidrotic '
      'ectodermal dysplasia with immunodeficiency (EDA-ID) (via TLR) pathway '
      'is a member of IKBKB deficiency causes SCID pathway.',
 'P1': 'Every member of IKBKG deficiency causes anhidrotic ectodermal '
       'dysplasia with immunodeficiency (EDA-ID) (via TLR) pathway is either a '
       'member of TLR3 deficiency - HSE pathway or a member of IKBKB '
       'deficiency causes SCID pathway, or both.',
 'P2': 'Whatever is a member of IKBKG deficiency causes anhidrotic ectodermal '
       'dysplasia with immunodeficiency (EDA-ID) (via TLR) pathway, is not a '
       'member of TLR3 deficiency - HSE pathway.'}


In [8]:
# Timed run over first N examples to estimate latency
import time, math
from tqdm import tqdm

N = min(10, len(items))  # number of examples to time
times = []
labels = []

def get_premises_and_hyp(ph_item):
    # collect premises P1..Pn in order
    pkeys = sorted([k for k in ph_item.keys() if k.startswith('P')], key=lambda x: int(x[1:]))
    premises = [ph_item[k] for k in pkeys]
    hyp = ph_item.get('C', '')
    # strip leading wrapper used in datasets
    hyp = hyp.replace('It is true that ', '').replace('It is false that ', '')
    return premises, hyp

print(f'Running {N} LangPro calls (this will make network requests)')
for i in range(N):
    premises, hyp = get_premises_and_hyp(items[i])
    t0 = time.perf_counter()
    try:
        out = langpro_api_call(premises, hyp, curl=False, report=False)
    except Exception as e:
        out = None
        print('Call failed on example', i, 'error:', e)
    t1 = time.perf_counter()
    elapsed = t1 - t0
    times.append(elapsed)
    labels.append(out['label'] if (out and 'label' in out) else None)
    print(f'[{i+1}/{N}] {elapsed:.2f}s ->', labels[-1])

avg = sum(times)/len(times)
std = math.sqrt(sum((t-avg)**2 for t in times)/len(times))
est_total_file = avg * len(items)

print('\nPer-call avg: %.2fs (std %.2fs)' % (avg, std))
print('Estimated time for this file (%.0f examples): %.1f seconds (%.1f minutes)' % (len(items), est_total_file, est_total_file/60.0))

# Estimate for all files
total_examples = 0
for fn in dataset_files:
    with open(os.path.join(DATASET_DIR, fn), 'r', encoding='utf-8') as f:
        total_examples += len(json.load(f))
est_all = avg * total_examples
print('Total examples across files:', total_examples)
print('Estimated time for all dataset files: %.1f hours' % (est_all/3600.0))

Running 10 LangPro calls (this will make network requests)
[1/10] 5.92s -> None
Call failed on example 1 error: name 'parse_kb' is not defined
[2/10] 6.64s -> None
[3/10] 4.84s -> None
Call failed on example 3 error: name 'parse_kb' is not defined
[4/10] 14.98s -> None
Call failed on example 4 error: name 'parse_kb' is not defined
[5/10] 13.87s -> None
[6/10] 4.86s -> None
Call failed on example 6 error: name 'parse_kb' is not defined
[7/10] 17.88s -> None
Call failed on example 7 error: name 'parse_kb' is not defined
[8/10] 18.03s -> None
Call failed on example 8 error: name 'parse_kb' is not defined
[9/10] 13.83s -> None
Call failed on example 9 error: name 'parse_kb' is not defined
[10/10] 12.85s -> None

Per-call avg: 11.37s (std 5.01s)
Estimated time for this file (200 examples): 2273.9 seconds (37.9 minutes)
Total examples across files: 26720
Estimated time for all dataset files: 84.4 hours


In [ ]:
# clone LangPro, a natural tableau prover. Use certain commit for stability
! git clone -q --branch nl https://github.com/kovvalsky/LangPro.git && cd LangPro && git checkout -q 4cdaa52

In [ ]:
# for directly displaying NLTK trees
!pip install -q svgling
!python3 -m pip show svgling | grep -i "version"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 4.0 MB/s eta 0:00:00
Version: 0.5.0


In [ ]:
import sys
from nltk import Tree, TreePrettyPrinter
# importing functions from langpro_api
sys.path.append("/content/LangPro/python")
from langpro_api import parse_ccg_tree, parse_term, \
                        parse_kb, tree_to_line, parse_info_proof

In [6]:
# utility function to call langpro API for NLI problems
def langpro_api_call(premises, hypothesis,
                     endpoint="https://langpro.hum.uu.nl/langpro-api/prove/",
                     parser="easyccg", ral=200, kb=[], senses = 'all',
                     strong_align=True, intersective=True, curl=False, report=False):
    """ Uses API call to a remote server to run LangPro prover
        and get parsed input sentecnes, tableau proof, and inference label.
        :param premises: list of premises
        :param hypothesis: hypothesis
        :param endpoint: endpoint of the server
        :param parser: CCG parser's name that will be used (default "easyccg")
        :param ral: rule applciation limit
        :param kb: user-injected knowledge base, a list of lexical relations
        :param senses: number of word senses used per word
        :param strong_align: whether to align indefinite NPs
        :param intersective: whether to treat modifiers by default as intersective
        :param curl: whether to print curl command
        :param report: whether to print error-related report
        :return: a dictionary with parsed input sentences, tableau proof, and inference label
    """
    import json, requests
    # preparing an input for the API call
    prob = {'premises': premises, 'hypothesis': hypothesis}
    headers={'Content-Type': 'application/json'}
    parameters = {  'prover_config': [],
                    'parser': parser,
                    'ral': ral,
                    'kb': kb,
                    'senses': senses    }
    if strong_align: parameters['prover_config'].append('aall')
    if intersective: parameters['prover_config'].append('allInt')
    query = {**prob, **parameters}
    js_query = json.dumps(query)

    # optinal, for curl command
    if curl:
        curl_command = f"curl '{endpoint}' " +\
        " ".join([f"-H '{k}: {v}'" for k, v in headers.items()]) +\
        f" -d '{js_query}'"
        print(curl_command)

    response = requests.post(endpoint, data=js_query, headers=headers)
    try:
        output = json.loads(response.text)
    except json.decoder.JSONDecodeError as e:
        if report:
            print(f"Failed to parse JSON response")
            print(f"Response status: {response.status_code}")
            print(f"Response headers: {response.headers}")
            print(f"Response text: {response.text[:20]}")  # First 20 chars
            print(f"Error: {e}")
        return None

    # parsing the components of the output
    kb = parse_kb(output['kb'])
    ccg_trees = [ parse_ccg_tree(i['tree']['ccg_tree']) for i in output['prob'] ]
    ccg_terms = [ parse_term(i['tree']['ccg_term']) for i in output['prob'] ]
    corr_terms = [ parse_term(i['tree']['corr_term']) for i in output['prob'] ]
    llfs = [ parse_term(i['tree']['llf']) for i in output['prob'] ]
    lab_proofs = { label: parse_info_proof(info_proof) \
                    for label, info_proof in output['proofs'].items() }
    # derive a predicted inference label
    entailment = 'closed' in output["proofs"]["entailment"]["info"]
    contradiction = 'closed' in output["proofs"]["contradiction"]["info"]
    if entailment and not contradiction:
        label = 'entailment'
    elif not entailment and contradiction:
        label = 'contradiction'
    else:
        label = 'neutral'
    # wrap up all in a dict
    return {'kb':kb, 'ccg':ccg_trees, 'ccg_terms':ccg_terms, 'label':label,
            'terms':corr_terms, 'llfs':llfs, 'proofs':lab_proofs}

# Using LangPro API

Let's use the LangPro API, allowing us to call LangPro from a remote server, without having it installed locally.
The API is a copy of this [LangPro demo](https://naturallogic.pro/LangPro/) that uses the C&C CCG parser (`parser="cc"`), rebanked version of the C&C parser (`parser="re-cc"`), and EasyCCG (`parser="easyccg"`).
EasyCCG is a good default choice. C&C has some technical issues and for now it is not recommended to use.

Though the API is a bit slow, it is easier to use than to install LangPro in Colab, which is [doable](https://colab.research.google.com/drive/10GjzWm6hO5yqjslYXrAjkasyM7L1s2B0?usp=sharing).


In [ ]:
# @title Simple example
# All animals sleep -> Every dog sleeps
output = langpro_api_call(["All animals sleep"],
                          "Every dog sleeps", curl=True)
print(f"Prediction = {output['label']}")
print(output.keys())

curl 'https://langpro.hum.uu.nl/langpro-api/prove/' -H 'Content-Type: application/json' -d '{"premises": ["All animals sleep"], "hypothesis": "Every dog sleeps", "prover_config": ["aall", "allInt"], "parser": "easyccg", "ral": 200, "kb": [], "senses": "all"}'
Prediction = entailment
dict_keys(['kb', 'ccg', 'ccg_terms', 'label', 'terms', 'llfs', 'proofs'])


In [ ]:
! curl 'https://langpro-annotator.hum.uu.nl/langpro-api/prove/' -H 'Content-Type: application/json' -d '{"premises": ["All animals sleep"], "hypothesis": "Every dog sleeps", "prover_config": ["aall", "allInt"], "parser": "easyccg", "ral": 200, "kb": [], "senses": "all"}'

{
  "aligned_llfs": "true",
  "kb": [
    {
      "args": [
        "dog",
        "animal"
      ],
      "functor": "isa_wn"
    }
  ],
  "prob": [
    {
      "role": "p",
      "sen": "All animals sleep",
      "sen_id": 1,
      "tree": {
        "ccg_term": {
          "args": [
            {
              "args": [
                {
                  "args": [
                    {
                      "args": [
                        "sleep",
                        "sleep",
                        "VBP",
                        "O",
                        "O"
                      ],
                      "functor": "tlp"
                    },
                    {
                      "args": [
                        {
                          "args": [
                            "np",
                            "_"
                          ],
                          "functor": ":"
                        },
                        {
                          "arg